## Schrodinger equation

In [ ]:
from solver import *
from psiop import *

t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Simple advection equation: u_t = -u_x  (symbol = i*xi)
# Note: psiOp(i*xi) corresponds to -d/dx
eq = Eq(diff(u, t), psiOp(-1j * xi, u)) 
solver = PDESolver(eq)

def initial_condition(x):
    return np.exp(-((x + 2)**2) / 0.5)  # Gaussian offset from center

solver = PDESolver(eq)
solver.setup(
    Lx=10.0, Nx=256, Lt=10.0, Nt=1000,
    boundary_condition='periodic',  # Sponge works with periodic BCs
    initial_condition=initial_condition,
    sponge=True,
    sponge_width=0.2,      # 20% of domain on each side
    sponge_strength=5.0,   # Moderate damping
    sponge_order=2,
)
solver.solve()

# Check if amplitude decreased
initial_amp = np.max(np.abs(solver.frames[0]))
final_amp = np.max(np.abs(solver.frames[-1]))
print(f"Initial amplitude: {initial_amp:.4f}")
print(f"Final amplitude: {final_amp:.4f}")
print(f"Absorption: {100*(1-final_amp/initial_amp):.1f}%")

plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # 'contour', 'front' or None
    mode='surface', # 'imshow' or 'surface'
    physical=True # True or False
)

HTML(ani.to_jshtml())

## Wave equation

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Wave equation
eq = Eq(diff(u, t, t), psiOp(-xi**2, u))
solver = PDESolver(eq)

def initial_condition(x):
    return np.exp(-(x / 0.3)**2)

def initial_velocity(x):
    return 0.0 * x

solver.setup(
    Lx=10.0, Nx=256,
    Lt=10.0, Nt=500,
    boundary_condition='periodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
    # ── Sponge layer ──
    sponge=True,
    sponge_width=0.1,  
    sponge_strength=5,  # max damping rate
    sponge_order=3,        # polynomial ramp order
)

frames = solver.solve()

# Check if amplitude decreased
initial_amp = np.max(np.abs(solver.frames[0]))
final_amp = np.max(np.abs(solver.frames[-1]))
print(f"Initial amplitude: {initial_amp:.4f}")
print(f"Final amplitude: {final_amp:.4f}")
print(f"Absorption: {100*(1-final_amp/initial_amp):.1f}%")

plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # 'contour', 'front' or None
    mode='surface', # 'imshow' or 'surface'
    physical=True # True or False
)

HTML(ani.to_jshtml())